# SA-Jev · 公开 LightRAG 索引与数据准备
默认使用三套公开 N500 图，不对训练段落重新付费建图。产物保存到 MyDrive/SA-Jev/public_lightrag_v1/pilot 或 full；旧 full 目录保留。full：8,000+4,000 道训练题；HotpotQA/MuSiQue 每库 300 测试、100 验证、100 扰动保留，2Wiki 500 测试。非训练题全部来自公开图配套清单，不表示 benchmark 全测试集。CPU 可运行，下载约 561 MB 索引和小型 MiniLM；不下载 OpenJev-4B。首次先跑 pilot；成功结束后默认刷新 Drive 并断开。

In [3]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Use 'full' after the pilot; separate Drive directories.
AUTO_DISCONNECT = True  # Release Colab after successful preparation and Drive flush.
from uuid import uuid4
PREPARATION_RUN_ID = uuid4().hex
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code: /content/graphrag-jev Artifacts: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full


In [4]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [5]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


Credential file: /content/drive/MyDrive/SA-Jev/.env
OPENAI_API_KEY: loaded
OPENAI_BASE_URL: loaded
TYPESAFE_API_KEY: loaded
HF_TOKEN: loaded


In [ ]:
import torch
from pathlib import Path
from tempfile import mkdtemp
from sa_jev.smoke import run

smoke_parent = ROOT / "smoke"
smoke_parent.mkdir(parents=True, exist_ok=True)

smoke_dir = Path(mkdtemp(prefix="run_", dir=smoke_parent))
print("本次测试目录：", smoke_dir)

summary = run(
    smoke_dir,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
summary

本次测试目录： /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full/smoke/run_hul_l63e
[data] Build trajectory preferences: starting 0/2; waiting for records...
[data] Build trajectory preferences: 1/2 records, 402.7/s, ETA=0s
{'step': 1, 'epoch': 0, 'rank': 1.7917581796646118, 'suf': 0.6931471228599548, 'rel': 0.6931471526622772, 'dpo': 0.0, 'stability': 0.0, 'paired_states': 0, 'context_pairs': 0, 'context_over_budget': 0, 'unpaired_states': 2, 'valid_stab_rank': 0, 'valid_stab_suf': 0, 'allocated_gib': 0.017200946807861328, 'reserved_gib': 0.041015625, 'peak_allocated_gib': 0.025798320770263672, 'peak_reserved_gib': 0.041015625, 'seconds': 2.1483701779999933, 'eta_seconds': 2.1483701779999933, 'grad_norm': 0.00011675283167278394, 'valid_rank': 1, 'valid_suf': 2, 'valid_rel': 10, 'task': 2.623534733057022, 'total_loss': 2.623534733057022}
{'step': 2, 'epoch': 0, 'rank': 1.6094295978546143, 'suf': 0.693147212266922, 'rel': 0.693133682012558, 'dpo': 0.0, 'stability': 0.0, 'paired_states': 0,

公开图的向量编码器为 all-MiniLM-L6-v2（384 维、256 token），与 BGE 不兼容。本地下载固定版本，仅用于查询和兼容性检查，不调用 embedding API。源发布未记录编码器 revision，加载图后用已存向量进行兼容性校验。

In [ ]:
import subprocess, importlib

subprocess.run(
    ['git', '-C', str(PROJECT), 'pull', '--ff-only'],
    check=True
)

from sa_jev import embeddings
importlib.reload(embeddings)
assert hasattr(embeddings, 'stage_embedding_snapshot'), '云端代码尚未更新'

embedding_check = await embeddings.check_embeddings(
    config['retrieval'], ROOT / 'embedding_check.json'
)
embedding_check

In [ ]:
# Retired one-time repair: zip_path_patch
# The current project already includes this fix. Original code is preserved in cell metadata.
print("Skipped obsolete repair; continuing with the current Git code.")


下载并校验公开 ZIP，按重复条目分别导入三套图；2Wiki 使用包内完整冲突副本，详见 public_manifest.json。复用旧 full/raw 下载（若完整），否则下载原始训练源。仅采样训练问题的段落用于 BM25 难例，不调用建图 API。公开配套题按问题族分为测试、验证和独立扰动保留题；不再从配套清单外抽题。旧划分首次运行时归档其派生数据与完成标记，保留图、原始下载和逐题候选缓存。少量缺失支持仍如实报告，不注入金标。

In [ ]:
import importlib
from sa_jev import embeddings, retrieval, public_indexes, preparation, experiments
for module in (embeddings, retrieval, public_indexes, preparation, experiments):
    importlib.reload(module)
config = experiments.load_config(CONFIG_PATH)
from sa_jev.public_indexes import download_public_assets, import_public_indexes, prepare_public_data
from sa_jev.preparation import download_sources
from sa_jev.io import read_json
archive = download_public_assets(PUBLIC_ASSETS)
index_manifest = import_public_indexes(archive, ROOT)
raw = ROOT/'raw'
legacy = DRIVE/'full/raw'
if MODE == 'full' and (legacy/'download_manifest.json').exists() and read_json(legacy/'download_manifest.json').get('formal'):
    raw = legacy
    print('Reusing existing source downloads:', raw)
download_sources(raw, limit=64 if MODE=='pilot' else None)
manifest = prepare_public_data(ROOT, PUBLIC_ASSETS, raw, config, mode=MODE)
manifest

In [ ]:
# Retired one-time repair: forced_minilm_redownload
# The current project already includes this fix. Original code is preserved in cell metadata.
print("Skipped obsolete repair; continuing with the current Git code.")


In [ ]:
# Retired one-time repair: manual_source_zip_overlay
# The current project already includes this fix. Original code is preserved in cell metadata.
print("Skipped obsolete repair; continuing with the current Git code.")


在导入的三个图上执行 LightRAG 原生 mix（图＋向量）召回，缓存最大 K=256，再切片得到主实验 K=32。保留已发布图，不重新抽取文档；查询关键词提取仍会调用 .env 中的 DeepSeek。完整协议有 1,500 个查询（1,100 测试、200 验证、200 独立保留），pilot 18 个。先检查 public_source_coverage.json，再执行检索；同题、同图、同配置的旧缓存可复用。初始缓存完成后，其余 notebook 复用候选。检查 public_pool_coverage.csv；缺失支持、未知标签仍掩蔽。

In [10]:
from sa_jev.experiments import cache_pools
await cache_pools(ROOT, config)

In [11]:
from sa_jev.preparation import prepare_tasks
prepare_tasks(ROOT/'data', counts=config['data']['train_counts'], mine=True, dense=config['data']['dense_mining'])

训练前完整扫描公开 OpenJevData。检出的重叠保留记录并在论文报告；未检出不代表证明完全未见。此步骤不使用独立扰动测试集调参。

In [12]:
cli('overlap')
cli('review-packets')

In [13]:
from sa_jev.preparation import finish_preparation
completion = finish_preparation(ROOT, config, mode=MODE, run_id=PREPARATION_RUN_ID)
completion

核对实际 K 分布和有效标签数量。缺候选不复制补齐，未知相关性不硬判负例。若排序有效状态太少，核查候选注释映射/补充核验，不能把掩码简单删掉。200 道保留题不等于 600 个合格状态包：实际草稿数量和独立题数见 perturbations_draft.manifest.json；只报告核验通过的实际数量。把独立扰动草稿人工核验后保存为 perturbations_reviewed.jsonl。

结束后自动断开：只有本次运行已成功完成才执行。先刷新并卸载 Drive，再释放 Colab 运行时。Drive 刷新失败或流程失败时不主动释放；重新连接需重装环境并挂载 Drive，已保存数据保留。本地运行会跳过此步骤。

In [ ]:
# from sa_jev.environment import disconnect_after_preparation
# disconnect_after_preparation(ROOT, run_id=PREPARATION_RUN_ID, enabled=AUTO_DISCONNECT)

In [15]:
from pathlib import Path
from collections import Counter, defaultdict
import json

ROOT = Path("/content/drive/MyDrive/SA-Jev/public_lightrag_v1/full")
DATA = ROOT / "data"

def rows(path):
    with path.open(encoding="utf-8-sig") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

required = ["questions.jsonl", "train_states.jsonl", "validation_states.jsonl"]
for name in required:
    assert (DATA / name).is_file(), f"找不到文件，请检查 Drive 是否已挂载：{DATA / name}"

# 只保留问题所属数据集，不把全部证据加载到内存。
datasets = {q["id"]: q["dataset"] for q in rows(DATA / "questions.jsonl")}

for filename in ("train_states.jsonl", "validation_states.jsonl"):
    totals = Counter()
    reasons = Counter()
    groups = defaultdict(Counter)
    unknown_examples = []

    print(f"\n开始审计：{filename}", flush=True)

    for i, item in enumerate(rows(DATA / filename), 1):
        s, y = item["state"], item["labels"]
        qid = s["question_id"]
        k = len(s["pool"]["candidates"])
        group = groups[(datasets.get(qid, "unknown"), k)]

        totals["状态总数"] += 1
        group["total"] += 1

        if y["rank_valid"]:
            reason = "排序标签有效"
            group["rank_valid"] += 1
        elif not y["gold_available"]:
            reason = "Gold 不完整或不可用"
        elif not y["suf_valid"]:
            reason = "当前充分性标签不可判定"
        elif y["sufficient"]:
            reason = "已经充分，正常屏蔽排序"
        elif sum(y["gains"]) <= 0:
            reason = "没有已知正增益候选"
        elif not all(y["rel_valid"]):
            reason = "存在正增益，但未知候选导致整体排序屏蔽"
        else:
            reason = "其他原因，需要检查标签一致性"
        reasons[reason] += 1

        # 单独统计尚未充分、Gold 和充分性标签可判定的状态。
        # 这里的两个子计数可能重叠，不要相加作为总数。
        if y["gold_available"] and y["suf_valid"] and not y["sufficient"]:
            totals["可判定且尚未充分的状态"] += 1
            group["needs_evidence"] += 1

            if sum(y["gains"]) <= 0:
                totals["其中：无已知正增益"] += 1

            if not all(y["rel_valid"]):
                totals["其中：含未知候选"] += 1
                group["unknown_candidates"] += 1

                if len(unknown_examples) < 5:
                    unknown_examples.append({
                        "question_id": qid,
                        "K": k,
                        "selected_count": len(s["selected_ids"]),
                        "remaining_candidates": len(y["candidate_ids"]),
                        "unknown_candidates": sum(not v for v in y["rel_valid"]),
                        "positive_gain_candidates": sum(g > 0 for g in y["gains"]),
                    })

        if i % 5000 == 0:
            print(f"已检查 {i:,} 个状态", flush=True)

    print("\n总览：")
    print(json.dumps(dict(totals), ensure_ascii=False, indent=2))
    print("\n排序有效／屏蔽原因（互斥分类）：")
    print(json.dumps(dict(reasons), ensure_ascii=False, indent=2))

    print("\n按数据集与初始 K 分组：")
    print("dataset | K | total | rank_valid | needs_evidence | unknown_candidates")
    for (dataset, k), c in sorted(groups.items()):
        print(f"{dataset} | {k} | {c['total']} | {c['rank_valid']} | "
              f"{c['needs_evidence']} | {c['unknown_candidates']}")

    print("\n含未知候选的状态示例：")
    print(json.dumps(unknown_examples, ensure_ascii=False, indent=2))

print("\n审计完成：未修改任何数据文件。")